# 01 — Construir el ABT (Databricks → UC + S3)
Lee el DW `semi2.dm_mortality`, arma el ABT (largo y ancho), lo materializa en tablas de Unity Catalog (`abt_ine`, `abt_ine_wide_pre/post`) y además lo exporta a `s3://mortality-analytics-semi2/ml/abt/` como Parquet para descargarlo localmente con `aws s3 sync`. El build se registra en MLflow para linaje.

In [ ]:
from pyspark.sql import functions as F
import boto3, os, mlflow

BUCKET = 'mortality-analytics-semi2'
S3_ABT = 'ml/abt'
CAT = 'semi2.dm_mortality'

mlflow.set_experiment('/Users/jorgemejia251@outlook.com/semi2_ml_mortality')


def _s3():
    return boto3.client(
        's3', region_name='us-east-1',
        aws_access_key_id=dbutils.secrets.get('semis2_scope', 'AWS_ACCESS_KEY_ID'),
        aws_secret_access_key=dbutils.secrets.get('semis2_scope', 'AWS_SECRET_ACCESS_KEY'))


def _to_s3_spark(df, key, fmt='parquet'):
    """Exporta un Spark DataFrame a S3 como un solo archivo (parquet/csv).
    patron identico a notebooks/phase#1/s3.py: boto3.upload_file desde /tmp."""
    os.makedirs('/tmp/ml_export', exist_ok=True)
    local = f"/tmp/ml_export/{key.replace('/', '_')}"
    pdf = df.toPandas()
    if fmt == 'parquet':
        pdf.to_parquet(local, index=False)
    else:
        pdf.to_csv(local, index=False)
    try:
        _s3().upload_file(local, BUCKET, f'{key}.{fmt}')
        print(f'  s3 -> s3://{BUCKET}/{key}.{fmt}')
    except Exception as e:
        print(f'[WARN] S3 no disponible: {e} (queda en {local})')


f   = spark.table(f'{CAT}.fact_ine')
t   = spark.table(f'{CAT}.dim_tiempo')
g   = spark.table(f'{CAT}.dim_geografia')
c   = spark.table(f'{CAT}.dim_causa')
gen = spark.table(f'{CAT}.dim_genero')
e   = spark.table(f'{CAT}.dim_etario')

## Capítulo CIE-10 (feature engineering)
Reduce 3,146 causas a capítulos manejables. **U07 = COVID** se separa aparte.

In [ ]:
def capitulo(col):
    return (F.when(col.startswith('U07'), 'COVID-19')
             .when(col.rlike('^[AB]'), 'Infecciosas')
             .when(col.rlike('^[CD]'), 'Neoplasias')
             .when(col.rlike('^E'),    'Endocrinas/Metabolicas')
             .when(col.rlike('^I'),    'Circulatorias')
             .when(col.rlike('^J'),    'Respiratorias')
             .when(col.rlike('^K'),    'Digestivas')
             .when(col.rlike('^[VWXY]'), 'Causas externas')
             .otherwise('Otras'))

## ABT largo (Modelos A y B)

**Forma larga = muchas filas, pocas columnas.** Una fila por *cada combinación* de
(mes, departamento, causa, sexo, edad), con `defunciones` como valor:

| departamento | causa_capitulo | defunciones |
|---|---|---|
| Guatemala | Respiratorias | 142 |
| Guatemala | Circulatorias | 200 |
| Guatemala | COVID-19 | 350 |

**Por qué esta forma:** la usan los modelos que predicen un **número**. Cada fila es un
*caso* con sus pistas (X) y su target (`defunciones`, y):
- **Modelo A (Regresión)** → analiza el **tiempo** → necesita una fila por mes.
- **Modelo B (Random Forest)** → analiza **cada caso** → necesita una fila por combinación.

In [ ]:
abt = (f
  .join(t, 'id_tiempo').join(g, 'id_geografia').join(c, 'id_causa')
  .join(gen, 'id_genero').join(e, 'id_etario')
  .where(F.col('dep_ocurrencia').isNotNull())
  .withColumn('causa_capitulo', capitulo(F.col('`cie-10_code`')))
  .groupBy(
     F.col('anio_ocurrencia').alias('anio'),
     F.col('mes_ocurrencia_num').alias('mes'),
     'es_pre_covid',
     # Normalizamos tildes: "Suchitepequez" y "Suchitepéquez" son el mismo depto.
     # Sin esto, pre sale con 23 departamentos en vez de 22.
     F.translate(F.col('dep_ocurrencia'), 'áéíóúÁÉÍÓÚ', 'aeiouAEIOU').alias('departamento'),
     'causa_capitulo',
     F.col('sexo_nombre').alias('sexo'),
     F.col('categoria_etaria').alias('grupo_edad'))
  .agg(F.sum('defuncion').alias('defunciones')))

abt.write.mode('overwrite').saveAsTable(f'{CAT}.abt_ine')
_to_s3_spark(abt, f'{S3_ABT}/abt_ine')

with mlflow.start_run(run_name='build_abt_largo'):
    mlflow.log_params({'grano': 'anio,mes,es_pre_covid,departamento,causa_capitulo,sexo,grupo_edad',
                       'target': 'defunciones'})
    mlflow.log_metric('filas', abt.count())

print('ABT largo:', abt.count(), 'filas -> UC +', f's3://{BUCKET}/{S3_ABT}/abt_ine.parquet')
display(abt.limit(10))

## ABT ancho (Modelo C / K-means)

**Forma ancha = pocas filas, muchas columnas.** Una fila por *departamento*, y cada causa
se vuelve **su propia columna** (`pct_<capitulo>`). Es el mismo dato del ABT largo "doblado"
con un *pivot*:

| departamento | pct_Respiratorias | pct_Circulatorias | pct_COVID-19 |
|---|---|---|---|
| Guatemala | 0.18 | 0.26 | 0.45 |
| Quetzaltenango | 0.28 | 0.42 | 0.10 |

**Por qué K-means necesita esta forma:** K-means **agrupa departamentos parecidos**, así que
necesita comparar departamentos *entre sí* → cada departamento debe ser **una sola fila** y su
"perfil" debe estar **en columnas** para poder medir distancias:

```
Guatemala      → [0.18, 0.26, 0.45]  ┐
Quetzaltenango → [0.28, 0.42, 0.10]  ├─ K-means mide distancia y junta los parecidos
Escuintla      → [0.17, 0.25, 0.47]  ┘
```

Si le diéramos el ABT *largo*, Guatemala estaría en varias filas (una por causa) y K-means no
lo vería como "un departamento con un perfil".

**Por qué proporciones (%) y no conteos:** Guatemala tiene muchas más muertes que un
departamento chico. Con conteos, K-means agruparía por **tamaño**; con **%** agrupa por la
*forma del perfil* ("de qué se muere la gente"), que es lo que queremos.

**Por qué `pre` y `post` por separado:** corremos K-means en cada periodo y comparamos qué
departamentos **cambiaron de grupo** tras el COVID → evidencia del cambio estructural.

In [ ]:
import re

def safe(nombre):
    # Delta no permite ' ,;{}()\n\t=' en nombres de columna -> los normalizamos
    return 'pct_' + re.sub(r'[ ,;{}()/\t=\-]+', '_', nombre)

for etiqueta, flag in [('pre', True), ('post', False)]:
    sub = abt.where(F.col('es_pre_covid') == flag)
    tot = sub.groupBy('departamento').agg(F.sum('defunciones').alias('tot'))
    piv = (sub.groupBy('departamento').pivot('causa_capitulo').sum('defunciones')
              .na.fill(0).join(tot, 'departamento'))
    caps = [col for col in piv.columns if col not in ('departamento', 'tot')]
    sel = ['departamento']
    for col in caps:
        piv = piv.withColumn(safe(col), F.col(f'`{col}`') / F.col('tot'))
        sel.append(safe(col))
    piv = piv.select(*sel)
    piv.write.mode('overwrite').saveAsTable(f'{CAT}.abt_ine_wide_{etiqueta}')
    _to_s3_spark(piv, f'{S3_ABT}/abt_ine_wide_{etiqueta}')
    with mlflow.start_run(run_name=f'build_abt_wide_{etiqueta}'):
        mlflow.log_params({'periodo': etiqueta, 'forma': 'ancho', 'features': 'pct_capitulos_cie10'})
        mlflow.log_metric('deptos', piv.count())
    print(f'ABT ancho {etiqueta} ->', piv.count(), 'departamentos -> UC +',
          f's3://{BUCKET}/{S3_ABT}/abt_ine_wide_{etiqueta}.parquet')
    display(piv)

## Verificación — las 3 tablas del ABT

Confirma que quedaron creadas y revisa su contenido antes de entrenar los modelos.

In [ ]:
for tabla in ['abt_ine', 'abt_ine_wide_pre', 'abt_ine_wide_post']:
    df = spark.table(f'semi2.dm_mortality.{tabla}')
    print(f'{tabla}: {df.count()} filas, {len(df.columns)} columnas')
    display(df.limit(5))